# 策略回测

本 Notebook 演示：
1. 加载因子数据，运行动量策略生成信号
2. 将信号写入 `signals.trading_signals`
3. 简单持仓模拟 + 绩效指标

In [1]:
import os
import sys
sys.path.insert(0, '/app')

import polars as pl
print(f'Polars version: {pl.__version__}')

Polars version: 1.39.0


In [2]:
# ── 1. 读取因子数据 ──────────────────────────────────────────
df_factors = pl.read_database_uri(
    query="""
        SELECT time, symbol, factor_name, factor_value
        FROM factors.daily_factors
        WHERE symbol = '000001.SZ'
          AND time >= '2023-01-01'
        ORDER BY time, factor_name
    """,
    uri=os.environ['DATABASE_URL'],
)
print(f'因子记录数: {len(df_factors)}')
df_factors.head()

因子记录数: 1936


time,symbol,factor_name,factor_value
"datetime[μs, UTC]",str,str,f64
2023-01-03 00:00:00 UTC,"""000001.SZ""","""ma20""",NaN
2023-01-03 00:00:00 UTC,"""000001.SZ""","""ma60""",NaN
2023-01-03 00:00:00 UTC,"""000001.SZ""","""macd""",NaN
2023-01-03 00:00:00 UTC,"""000001.SZ""","""rsi14""",NaN
2023-01-04 00:00:00 UTC,"""000001.SZ""","""ma20""",NaN


In [3]:
# ── 2. 运行动量策略生成信号 ──────────────────────────────────
from app.strategy.momentum import MomentumStrategy

strategy = MomentumStrategy(rsi_overbought=70, rsi_oversold=30)
df_signals = strategy.generate_signals(df_factors, universe=['000001.SZ'])
df_signals.filter(pl.col('signal') != 0)

2026-03-17 12:53:48.682 | INFO     | app.strategy.momentum:generate_signals:89 - [momentum_v1] 生成信号 425 条，买入=4，卖出=7


time,symbol,strategy,signal,score,metadata
"datetime[μs, UTC]",str,str,i32,f64,str
2023-08-09 00:00:00 UTC,"""000001.SZ""","""momentum_v1""",1,0.0525,null
2023-09-04 00:00:00 UTC,"""000001.SZ""","""momentum_v1""",-1,-0.008,null
2024-02-20 00:00:00 UTC,"""000001.SZ""","""momentum_v1""",1,0.012,null
2024-02-22 00:00:00 UTC,"""000001.SZ""","""momentum_v1""",-1,0.148,null
2024-06-24 00:00:00 UTC,"""000001.SZ""","""momentum_v1""",-1,-0.0145,null
…,…,…,…,…,…
2024-09-13 00:00:00 UTC,"""000001.SZ""","""momentum_v1""",-1,-0.007,null
2024-09-30 00:00:00 UTC,"""000001.SZ""","""momentum_v1""",-1,0.033833,null
2024-10-08 00:00:00 UTC,"""000001.SZ""","""momentum_v1""",-1,0.121333,null


In [ ]:
# ── 3. 写入 signals.trading_signals ─────────────────────────
from app.utils.signals import upsert_signals

n = upsert_signals(df_signals)
print(f'写入信号数: {n}')

In [5]:
# ── 4. 简单回测：按信号模拟每日收益 ─────────────────────────
df_price = pl.read_database_uri(
    query="""
        SELECT time, symbol, pct_change
        FROM market.daily
        WHERE symbol = '000001.SZ'
          AND time >= '2023-01-01'
        ORDER BY time
    """,
    uri=os.environ['DATABASE_URL'],
)

# 用前一日信号乘以当日收益（次日开盘执行）
df_bt = (
    df_signals
    .select(['time', 'symbol', 'signal'])
    .join(df_price, on=['time', 'symbol'])
    .sort('time')
    .with_columns(
        (pl.col('signal').shift(1) * pl.col('pct_change') / 100).alias('strategy_ret')
    )
    .drop_nulls('strategy_ret')
)
df_bt.tail(10)

time,symbol,signal,pct_change,strategy_ret
"datetime[μs, UTC]",str,i32,"decimal[38,10]","decimal[38,10]"
2024-12-18 00:00:00 UTC,"""000001.SZ""",-1,1.0408000000,0.0000000000
2024-12-19 00:00:00 UTC,"""000001.SZ""",0,-0.5150000000,0.0051500000
2024-12-20 00:00:00 UTC,"""000001.SZ""",0,0.2588000000,0.0000000000
2024-12-23 00:00:00 UTC,"""000001.SZ""",0,0.9466000000,0.0000000000
2024-12-24 00:00:00 UTC,"""000001.SZ""",0,1.1083000000,0.0000000000
2024-12-25 00:00:00 UTC,"""000001.SZ""",0,0.5059000000,0.0000000000
2024-12-26 00:00:00 UTC,"""000001.SZ""",0,-0.5034000000,0.0000000000
2024-12-27 00:00:00 UTC,"""000001.SZ""",0,-0.2530000000,0.0000000000
2024-12-30 00:00:00 UTC,"""000001.SZ""",1,1.0144000000,0.0000000000


In [6]:
# ── 5. 绩效指标 ──────────────────────────────────────────────
from app.backtest.metrics import compute_metrics

metrics = compute_metrics(df_bt['strategy_ret'], freq='daily')
for k, v in metrics.items():
    print(f'{k:20s}: {v}')

annualized_return   : 0.0725
annualized_vol      : 0.1156
sharpe              : 0.6272
max_drawdown        : -0.0665
calmar              : 1.091
